# 00 — Project status

Dashboard dello stato corrente: ambiente, manifest dati, gate della pipeline v2, artefatti presenti. Solo lettura: nessuna cella modifica lo stato del progetto.

Fonti di verità: `CLAUDE.md` (checklist), `experiments/decisions.md` (motivazioni), `CHANGELOG.md` (stato file per file).

In [ ]:
# Provenance header: every figure must be traceable to the state that made it.
import json, subprocess, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

def sh(*cmd):
    try:
        return subprocess.check_output(cmd, cwd=ROOT, text=True).strip()
    except Exception as e:
        return f"<unavailable: {e}>"

GIT_COMMIT = sh("git", "rev-parse", "--short", "HEAD")
GIT_DIRTY = bool(sh("git", "status", "--porcelain"))
MANIFEST_PATH = ROOT / "data/processed/manifest.json"
MANIFEST = json.loads(MANIFEST_PATH.read_text()) if MANIFEST_PATH.exists() else None
print(f"commit={GIT_COMMIT} dirty={GIT_DIRTY}")
print(f"manifest: {'present, created ' + MANIFEST['created_at'] if MANIFEST else 'ABSENT'}")


In [ ]:
# Environment
import torch, transformers, peft, datasets, sklearn
for m in (torch, transformers, peft, datasets, sklearn):
    print(f"{m.__name__:14s} {m.__version__}")
print("cuda available:", torch.cuda.is_available(), "(atteso: False, ambiente CPU)")


In [ ]:
# Dataset manifest -> tabella split
import pandas as pd
if MANIFEST:
    rows = []
    for name, s in MANIFEST["splits"].items():
        rows.append({"split": name, "n": s["n"], "mcq": s["n_mcq"],
                     **{f"type_{k}": v for k, v in s["by_type"].items()},
                     "sources": ", ".join(f"{k}:{v}" for k, v in s["by_source"].items())})
    display(pd.DataFrame(rows).set_index("split"))
    print("overlap (0 ovunque tranne probe, che è vista derivata di eval+test):")
    print(json.dumps(MANIFEST["overlap"], indent=2))
    print("seed:", MANIFEST["seed"], "| algo:", MANIFEST["split_algorithm"],
          "| built at commit:", MANIFEST["git"]["commit"][:9])


In [ ]:
# Gate della pipeline v2 — aggiornare A MANO quando un gate cambia stato,
# citando la voce di decisions.md che lo documenta.
import pandas as pd
gates = [
    ("test suite verde (gate 0)",        "PASS", "51 passed (44 unit + 7 integration), CI verde"),
    ("review data_contract/bottleneck",  "PASS", "decisions 2026-07-15: 5 fix + mask verificata su Qwen"),
    ("build dati v2 (fixture+reale)",    "PASS", "decisions 2026-07-15: manifest, zero overlap"),
    ("toy code-recall gate",             "PASS", "tent.2: acc 0.925 unseen, swap 0.90"),
    ("integrazione bottleneck end-to-end (P0)", "PASS", "percorso condiviso + mask-spy"),
    ("ri-pin criterio gating Exp 2",     "PASS", "541 MCQ, McNemar, 8 gating + 2 diagnostiche"),
    ("Exp 1b stabilita'",                "PASS", "ppl +0.24% (v0: +24.7%) su n=500"),
    ("Exp 0 v2 baseline",                "DONE", "summary 0.656, full 0.808; effetto bigino v0 = rumore"),
    ("Exp 2 recall vs baseline",         "FAIL", "bottleneck 0.237 vs 0.656 (p=3.9e-46); capsula ~ vuota; early stop dichiarato su 5 condizioni"),
    ("Exp 3 probing capsula",            "NEXT", "informazione presente ma inutilizzata, o assente?"),
]
display(pd.DataFrame(gates, columns=["gate", "stato", "riferimento"]))


In [ ]:
# Artefatti risultati presenti
for p in sorted((ROOT / "results").glob("*.json")):
    data = json.loads(p.read_text())
    keys = ", ".join(list(data)[:6])
    print(f"{p.name:34s} keys: {keys}...")
